# Wheat Data Collector

This notebook combines web scraping of wheat-related news and preprocessing to extract dense embeddings and sentiment scores using FinBERT.

## Pipeline Overview
1. **Scraping**: Fetches historical and current news articles related to wheat from sources like Investing.com and SERP API.
2. **Cleaning & Deduplication**: Cleans the text and drops duplicate news based on the article's source and headline.
3. **Embedding Extraction**: Extracts 768-d `[CLS]` embeddings using `ProsusAI/finbert`.
4. **Dimensionality Reduction**: Applies PCA to reduce the embeddings from 768-d to 16-d.
5. **Sentiment Scoring**: Uses the FinBERT sentiment classification head to get positive/negative/neutral probabilities.
6. **Temporal Alignment**: Aggregates the embeddings and sentiment scores by calendar day.

Outputs are saved in the `data/` directory:
- `data/wheat_news.csv`: Cleaned news articles.
- `data/wheat_news_sentiment.csv`: Daily aggregated sentiment scores.
- `data/wheat_news_embeddings.pt`: Daily aggregated PCA-reduced embeddings.

In [6]:
import os
import re
import sys
import time
import random
import logging
from pathlib import Path

import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from serpapi import GoogleSearch
from tqdm.auto import tqdm

import torch
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
from sklearn.decomposition import PCA

# Setup Logging & Environment
load_dotenv()
logging.basicConfig(level=logging.WARNING)

# Constants & Configurations
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

OUT_CSV = DATA_DIR / 'wheat_news.csv'
OUTPUT_PT = DATA_DIR / 'wheat_news_embeddings.pt'
OUTPUT_CSV = DATA_DIR / 'wheat_news_sentiment.csv'

# Scraping configs
INVESTING_CONFIGS = [
    {
        'name': 'wheat',
        'base_url': 'https://www.investing.com/commodities/us-wheat-news',
        'last_page': 46,
    }
]

SERP_API_KEY = os.getenv('SERP_API')
if not SERP_API_KEY:
    print("WARNING: SERP_API key not found in .env file. SERP scraping will fail if attempted.")

SERP_NEWS_SOURCES = [  
    "bloomberg.com",
    "reuters.com",                
    "cnbc.com",
    "investing.com"
]

SERP_COMMODITIES = [
    {
        'name': 'wheat', 
        'query': 'wheat AND ("prices" OR "supply" OR "exports" OR "futures" OR "market") AND ("war" OR "invasion" OR "tariff" OR "trade war" OR "sanctions" OR "covid" OR "pandemic" OR "drought" OR "crisis" OR "shock")'
    },
]

# Scraper settings
DELAY_MIN = 2.0
DELAY_MAX = 4.5
CHECKPOINT = 10
MAX_RETRIES = 3

# FinBERT Configurations
MODEL_NAME = "ProsusAI/finbert"
MAX_LEN    = 512
BATCH_SIZE = 8
PCA_DIM    = 16

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


## Part 1: News Scraper Helpers
Functions for fetching data from Investing.com and SERP API, and storing records to CSV.

In [7]:
# ═══ INVESTING.COM SCRAPER ═══
def make_scraper():
    """Create a cloudscraper session that mimics Chrome on macOS."""
    return cloudscraper.create_scraper(
        browser={'browser': 'chrome', 'platform': 'darwin', 'mobile': False}
    )

def page_url(base_url: str, page: int) -> str:
    if page == 1:
        return base_url
    return f'{base_url}/{page}'

def fetch_page(scraper, url: str, retries: int = MAX_RETRIES):
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-US,en;q=0.9',
        'Referer': 'https://www.investing.com/',
    }
    for attempt in range(1, retries + 1):
        try:
            r = scraper.get(url, headers=headers, timeout=30)
            if r.status_code == 200:
                return BeautifulSoup(r.text, 'lxml')
            elif r.status_code == 404:
                return None
            else:
                print(f'    HTTP {r.status_code} on attempt {attempt}: {url}')
        except Exception as e:
            print(f'    Error attempt {attempt}: {e}')
        if attempt < retries:
            wait = 2 ** attempt + random.uniform(0, 2)
            time.sleep(wait)
    return None

def parse_articles_investing(soup, commodity: str, page: int) -> list[dict]:
    records = []
    for art in soup.find_all('article', attrs={'data-test': 'article-item'}):
        title_tag = art.find('a', attrs={'data-test': 'article-title-link'})
        title = title_tag.get_text(strip=True) if title_tag else ''
        url = title_tag['href'] if title_tag else ''
        if url and not url.startswith('http'):
            url = 'https://www.investing.com' + url

        desc_tag = art.find('p', attrs={'data-test': 'article-description'})
        description = desc_tag.get_text(strip=True) if desc_tag else ''

        src_tag = art.find('a', attrs={'data-test': 'article-provider-link'})
        source = src_tag.get_text(strip=True) if src_tag else 'investing.com'

        date_tag = art.find('time', attrs={'data-test': 'article-publish-date'})
        if date_tag:
            date = date_tag.get('datetime', date_tag.get_text(strip=True))
        else:
            date = ''

        if title: 
            records.append({
                'commodity': commodity,
                'title': title,
                'date': date,
                'source': source,
                'description': description,
                'url': url,
            })
    return records

def already_scraped_pages(csv_path: Path) -> set[int]:
    if csv_path.exists():
        try:
            df = pd.read_csv(csv_path)
            if 'page_scraped' in df.columns:
                return set(df['page_scraped'].dropna().astype(int).tolist())
        except Exception:
            pass
    return set()

# ═══ SERP API SCRAPER ═══
def fetch_commodity_news_serp(commodity_name: str, query: str, limit_per_year: int = 50) -> list[dict]:
    records = []
    years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
    
    for source in SERP_NEWS_SOURCES:
        search_query = f'site:{source} {query}'
        
        for year in years:
            year_records = []
            start = 0
            
            while len(year_records) < limit_per_year:
                params = {
                    "q": search_query,
                    "api_key": SERP_API_KEY,
                    "num": 100,
                    "start": start,
                    "tbs": f"cdr:1,cd_min:01/01/{year},cd_max:12/31/{year}" 
                }
                
                try:
                    search = GoogleSearch(params)
                    results = search.get_dict()
                    
                    if "organic_results" not in results or not results["organic_results"]:
                        break
                        
                    for result in results["organic_results"]:
                        if len(year_records) >= limit_per_year:
                            break
                        
                        year_records.append({
                            'commodity': commodity_name,
                            'title': result.get('title', ''),
                            'date': result.get('date', ''),
                            'source': source,
                            'description': result.get('snippet', ''),
                            'url': result.get('link', ''),
                        })
                    
                    start += 100 
                    
                except Exception as e:
                    print(f"✗ Error fetching from {source} for year {year}: {str(e)}")
                    break
            
            if year_records:
                records.extend(year_records)
                print(f"✓ Fetched {len(year_records)} articles from {source} ({year})")
    
    return records

def append_to_csv(records: list[dict], csv_path: Path):
    if not records:
        return
    try:
        df_new = pd.DataFrame(records)
        write_header = not csv_path.exists()
        df_new.to_csv(csv_path, mode='a', header=write_header, index=False)
    except Exception as e:
        print(f"  [append_to_csv] ERROR: {str(e)}")

def normalize_source(source):
    source_lower = str(source).lower()
    if 'bloomberg' in source_lower:
        return 'Bloomberg'
    elif 'reuters' in source_lower:
        return 'Reuters'
    elif 'cnbc' in source_lower:
        return 'CNBC'
    elif 'investing' in source_lower:
        return 'Investing.com'
    return source

## Part 2: Main Scraping & Deduplication Function
Orchestrates scraping from both sources and deduplicates the results.
Crucially, it drops duplicate news based on **source and headline (title)**.

In [ ]:
def scrape_commodity(config: dict, out_csv: Path):
    name      = config['name']
    base_url  = config['base_url']
    last_page = config['last_page']

    done_pages  = already_scraped_pages(out_csv)
    todo_pages  = [p for p in range(1, last_page + 1) if p not in done_pages]

    if not todo_pages:
        print(f'[{name}] All {last_page} pages already scraped. CSV: {out_csv}')
        return

    print(f'[{name}] Scraping {len(todo_pages)} pages (skipping {len(done_pages)} already done)…')
    scraper  = make_scraper()
    buffer   = []

    for i, page in enumerate(tqdm(todo_pages, desc=name, unit='page'), start=1):
        url  = page_url(base_url, page)
        soup = fetch_page(scraper, url)

        if soup is not None:
            records = parse_articles_investing(soup, name, page)
            buffer.extend(records)

        if i % CHECKPOINT == 0 and buffer:
            append_to_csv(buffer, out_csv)
            buffer.clear()

        if i < len(todo_pages):
            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

    if buffer:
        append_to_csv(buffer, out_csv)

def run_scraper_and_clean():
    print("="*60)
    print("STARTING DUAL-SOURCE NEWS SCRAPING FOR WHEAT FUTURES")
    print("="*60 + "\n")

    # PHASE 1: Investing.com
    print("[PHASE 1/2] Scraping Investing.com...")
    for config in INVESTING_CONFIGS:
        scrape_commodity(config, OUT_CSV)

    # PHASE 2: SERP API
    if SERP_API_KEY:
        print("\n[PHASE 2/2] Scraping SERP API financial news sources...\n")
        all_serp_records = []
        for i, config in enumerate(SERP_COMMODITIES, start=1):
            print(f"[{i}/{len(SERP_COMMODITIES)}] Fetching news for {config['name'].upper()}...")
            records = fetch_commodity_news_serp(config['name'], config['query'])
            all_serp_records.extend(records)
        
        if all_serp_records:
            append_to_csv(all_serp_records, OUT_CSV)
            print(f"✓ SERP API: {len(all_serp_records)} total articles appended.")

    # PHASE 3: Cleaning & Deduplication
    print("\n[PHASE 3] Cleaning and deduplicating dataset based on source and headline...")
    if OUT_CSV.exists():
        df_final = pd.read_csv(OUT_CSV)
        print(f"Initial count: {len(df_final)} articles")
        
        df_final['commodity'] = 'wheat'
        df_final['source'] = df_final['source'].apply(normalize_source)
        
        # Parse dates robustly
        df_final['date'] = pd.to_datetime(df_final['date'], errors='coerce', format='mixed')
        df_final = df_final.dropna(subset=['date'])
        
        # Sort by date
        df_final = df_final.sort_values('date', ascending=False).reset_index(drop=True)
        
        # IMPORTANT: Deduplicate based on source and title (headline)
        initial_count = len(df_final)
        df_final = df_final.drop_duplicates(subset=['source', 'title'], keep='first')
        print(f"After deduplication by source & headline: {len(df_final)} articles (removed {initial_count - len(df_final)} duplicates)")
        
        required_cols = ['commodity', 'title', 'date', 'source', 'description', 'url']
        df_final = df_final[required_cols]
        
        # Save back to OUT_CSV
        df_final.to_csv(OUT_CSV, index=False)
        print(f"✓ FINAL: {len(df_final)} unique wheat articles saved to {OUT_CSV}")
        return df_final
    else:
        print(f"ERROR: No data found in {OUT_CSV}.")
        return pd.DataFrame()

df_news = run_scraper_and_clean()

## Part 3: FinBERT Embeddings & Sentiment Pipeline
Takes the `wheat_news.csv`, extracts FinBERT embeddings and sentiments, applies PCA, and aggregates by day.

In [9]:
_BOILERPLATE_RE = re.compile(
    r"^"
    r"(?:\*\s*)?"
    r"(?:"
        r"By\s+[A-Z][a-zA-Z\s\-']+"
        r"[A-Z]{2,}[\w\s,]*"
        r"\([^)]+\)"
        r"\s*[-–—]\s*"
    r")?"
    r"(?:\*\s*)?",
    re.MULTILINE,
)

def clean_text(row: pd.Series) -> str:
    title = str(row.get("title", "")).strip()
    description = str(row.get("description", "")).strip()
    
    description = _BOILERPLATE_RE.sub("", description).strip()
    if description.endswith("..."):
        description = description[:-3].strip()
        
    if description:
        return f"{title}. {description}"
    return title

def extract_cls_embeddings(texts: list[str], tokenizer, model, device, batch_size=BATCH_SIZE) -> torch.Tensor:
    all_embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Extracting embeddings"):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            outputs = model(**encoded)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(cls_embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

def apply_pca(embeddings: torch.Tensor, n_components=PCA_DIM) -> torch.Tensor:
    print(f"[INFO] Applying PCA: {embeddings.shape[1]} → {n_components} dimensions")
    embeddings_np = embeddings.numpy()
    pca = PCA(n_components=n_components)
    reduced_embeddings_np = pca.fit_transform(embeddings_np)
    print(f"[INFO] Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")
    return torch.from_numpy(reduced_embeddings_np).float()

def extract_sentiment_scores(texts: list[str], tokenizer, sentiment_model, device, batch_size=BATCH_SIZE) -> pd.DataFrame:
    all_scores = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Scoring sentiment"):
        batch_texts = texts[start : start + batch_size]
        encoded = tokenizer(
            batch_texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            logits = sentiment_model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).cpu()
            
        for row in probs:
            pos, neg, neu = row[0].item(), row[1].item(), row[2].item()
            all_scores.append({
                "sentiment_pos": pos,
                "sentiment_neg": neg,
                "sentiment_neu": neu,
                "sentiment_score": pos - neg,
            })
    return pd.DataFrame(all_scores)

def aggregate_daily(df: pd.DataFrame, embeddings: torch.Tensor) -> dict[str, torch.Tensor]:
    dates = df["date_day"].tolist()
    daily_map = {}
    for idx, d in enumerate(dates):
        daily_map.setdefault(d, []).append(idx)
        
    result = {}
    for day, indices in sorted(daily_map.items()):
        stacked = embeddings[indices]
        result[day] = torch.mean(stacked, dim=0)
    return result

## Part 4: Execute Embeddings & Sentiment Pipeline
Loads the scraped news, runs the text through FinBERT models, and outputs daily aggregated representations.

In [ ]:
def run_embedding_pipeline():
    print(f"[INFO] Loading news from {OUT_CSV}")
    if not OUT_CSV.exists():
        print(f"Error: {OUT_CSV} not found. Please run the scraper first.")
        return
        
    df = pd.read_csv(OUT_CSV)
    print(f"[INFO] Loaded {len(df)} articles")

    # Additional duplicate drop just to be perfectly safe
    n_before = len(df)
    df = df.drop_duplicates(subset=["title", "source"], keep="first")
    n_after = len(df)
    if (n_before - n_after) > 0:
        print(f"[INFO] Removed {n_before - n_after} duplicate(s) with same title & source")
        
    df = df.reset_index(drop=True)
    df["date"] = pd.to_datetime(df["date"])
    df["date_day"] = df["date"].dt.strftime("%Y-%m-%d")
    df["clean_text"] = df.apply(clean_text, axis=1)

    texts = df["clean_text"].tolist()

    print(f"[INFO] Loading tokenizer & model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME)
    model.to(device)
    model.eval()

    embeddings = extract_cls_embeddings(texts, tokenizer, model, device, BATCH_SIZE)
    embeddings = apply_pca(embeddings, PCA_DIM)

    print(f"[INFO] Loading FinBERT sentiment classification head")
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
    sentiment_model.to(device)
    sentiment_model.eval()

    sent_df = extract_sentiment_scores(texts, tokenizer, sentiment_model, device, BATCH_SIZE)
    df = pd.concat([df.reset_index(drop=True), sent_df], axis=1)
    
    # Cleanup memory
    del sentiment_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    daily_embeddings = aggregate_daily(df, embeddings)
    print(f"[INFO] {len(df)} articles aggregated into {len(daily_embeddings)} daily vectors")

    daily_sent = (df.groupby("date_day")[["sentiment_pos", "sentiment_neg", "sentiment_neu", "sentiment_score"]]
                    .mean()
                    .reset_index()
                    .rename(columns={"date_day": "date"}))
    daily_sent["article_count"] = df.groupby("date_day").size().values
    daily_sent = daily_sent.sort_values("date").reset_index(drop=True)

    torch.save(daily_embeddings, OUTPUT_PT)
    daily_sent.to_csv(OUTPUT_CSV, index=False)
    
    print("\n" + "═" * 60)
    print(f"  DONE — {len(daily_embeddings)} daily embeddings ({PCA_DIM}-d after PCA)")
    print(f"       — {len(daily_sent)} daily sentiment scores")
    print(f"  Outputs:    {OUTPUT_PT}\n              {OUTPUT_CSV}")
    print("═" * 60)

run_embedding_pipeline()

[INFO] Loading news from data/wheat_news.csv
[INFO] Loaded 1593 articles
[INFO] Removed 7 duplicate(s) with same title & source
[INFO] Loading tokenizer & model: ProsusAI/finbert


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting embeddings:   0%|          | 0/199 [00:00<?, ?it/s]

[INFO] Applying PCA: 768 → 16 dimensions
[INFO] Explained variance ratio: 0.7950
[INFO] Loading FinBERT sentiment classification head


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scoring sentiment:   0%|          | 0/199 [00:00<?, ?it/s]

[INFO] 1586 articles aggregated into 1293 daily vectors

════════════════════════════════════════════════════════════
  DONE — 1293 daily embeddings (16-d after PCA)
       — 1293 daily sentiment scores
  Outputs:    data/wheat_news_embeddings.pt
              data/wheat_news_sentiment.csv
════════════════════════════════════════════════════════════
